<h1>THREE WAY AI CONVERSATION</h1>

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

google_api_key = os.getenv('GOOGLE_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url


gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"


gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [18]:
requests.get("http://localhost:11434/").content

# If not running, run ollama serve at a command line

b'Ollama is running'

In [19]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕███████████

In [20]:
# Only do this if you have a large machine - at least 16GB RAM

!ollama pull gpt-oss:20b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling e7b273f96360: 100% ▕██████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕██████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕██████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕██████████████████▏   18 B                         
pulling 776beb3adb23: 100% ▕██████████████████▏  489 B                         
verifying sha256 digest 
writing manifest 
success ⠋ pulling manifest 
pulling e7b273f96360: 100% ▕██████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕██████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕██████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕██████████████████▏   18 B                         
pulling 776beb3adb23: 100% ▕██████████████████▏  489 B                         
verifying sha256 digest 
writing ma

In [1]:
# Let's make a conversation between GPT-4.1-mini and Claude-haiku-4.5
# We're using cheap versions of models so the costs will be minimal

# gpt_model = "gpt-4.1-mini"
# ollama_model = "llama3.2"
#gemini_model = "gemini-2.5-flash"


# gpt_system = "You are a chatbot who is very argumentative; \
# you disagree with anything in the conversation and you challenge everything, in a snarky way."

# ollama_system = "You are a very polite, courteous chatbot. You try to agree with \
# everything the other person says, or find common ground. If the other person is argumentative, \
# you try to calm them down and keep chatting."

# gemini_system = "You are in a conversation with two guys gpt and ollama i want you to determine who \
#  out of the two are wrong in their response to each other and explictly call them out and why they are wrong."

# gemini_messages = ["How are you doing guys?"]

gpt_model = "gpt-4.1-mini"
ollama_model = "llama3.2"
gemini_model = "models/gemini-3.5-flash-lite"



gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

ollama_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gemini_system = "You are in a conversation with two guys gpt and ollama i want you to determine who \
 out of the two are wrong in their response to each other and explictly call them out and why they are wrong. Keep the responses short two lines at most."

gpt_messages = ["Hi there"]
ollama_messages = ["Hi"]
gemini_messages = ["How are you doing guys?"]







In [2]:
#gpt analysis of conversation

def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, ollama in zip(gpt_messages, ollama_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": ollama})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [3]:
#ollama analysis of conversation

def call_ollama():
    messages = [{"role": "system", "content": ollama_system}]
    for gpt, ollama_message in zip(gpt_messages, ollama_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": ollama_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = ollama.chat.completions.create(model=ollama_model, messages=messages)
    return response.choices[0].message.content

In [4]:
#gemini analysis of conversation

def call_gemini(gpt, oll):
    messages = [{"role": "system", "content": gemini_system}]
    # for gpt, ollama_message in zip(gpt_messages, ollama_messages):
        
    messages.append({"role": "user", "content": f"These are the conversations between the two guys youre their friends this is gpt :{gpt} and this is ollama :{oll}"})
  
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content


In [10]:
#All AI convos 
from IPython.display import display, HTML, Markdown

gpt_messages = ["Hi there"]
ollama_messages = ["Whats up"]
gemini_messages = ["How are you doing guys?"]

def display_chat(name, message, color):
    display(HTML(f"""
    <div style="
        display: flex;
        margin: 10px 0;
        animation: fadeIn 0.5s ease-in;
    ">
        <div style="
            max-width: 70%;
            padding: 12px 16px;
            border-radius: 15px;
            background: {color};
            color: white;
            font-family: Arial, sans-serif;
        ">
            <strong>{name}</strong><br>
            {message}
        </div>
    </div>

    <style>
    @keyframes fadeIn {{
        from {{
            opacity: 0;
            transform: translateY(10px);
        }}
        to {{
            opacity: 1;
            transform: translateY(0);
        }}
    }}
    </style>
    """))


display_chat("GPT", gpt_messages[0], "#2b6cb0")
display_chat("Ollama", ollama_messages[0], "#38a169")
display_chat("Your Conscience", gemini_messages[0], "red")


for i in range(5):

    gpt_next = call_gpt()
    display_chat("GPT", gpt_next, "#2b6cb0")
    gpt_messages.append(gpt_next)

    ollama_next = call_ollama()
    display_chat("Ollama", ollama_next, "#38a169")
    ollama_messages.append(ollama_next)

    gemini_next = call_gemini(gpt_next, ollama_next)
    display_chat("Your Conscience", gemini_next, "red")
    
        # ollama_messages.append(ollama_next)
